# Generate Head Boundary Conditions for 2D transect ATS

Head extracted from Sundar's 3D ATS simulation

- file `OakCreek_from_sundar/bc_startend_raw.h5`
    - the complete folder in NERSC/$CFS/m1800/xiaoyi/OakCreek_from_sundar, which is more than 400GB
    - bc_startend_raw.h5 is the extracted water head providing coordinates of startpt and endpt
- Three datasets in this h5 file
    - "Time": from 11224 to 16059; unit is day;
        - 11224 = 274+365x30 --> 2010.10.1
        - 16059 = 364+365x43 --> 2023.12.31
    - "startpt_head"
    - "endpt_head"

Output of this script

- constant head at the starting and end points -> to drive the run0
- head at a typical year at the starting and end points -> to drive the run1
- transient head

**File History**

update 2025/10/13
- update `config.json`. Mainly revise the model run pipeline.

update 2025/8/32
- add `config.json`

update 2025/8/7
- correct site name to NF01
- a cleaned version putting all input data and notebooks together

In [ ]:
%load_ext autoreload
%autoreload 2

# Parameters and data sources

In [ ]:
# Parameters cell
import json
with open('config.json', 'r') as f:
    config = json.load(f)
watershed_name = config['watershed_name']
# hucs           = [config['hucs']]
site_name      = config['site_name']

# simulation control
start_year_spinup         = config['start_year_spinup']
end_year_spinup           = config['end_year_spinup']
nyears_steadystate_spinup = config['nyears_steadystate_spinup']
nyears_cyclic_spinup      = config['nyears_cyclic_spinup']
start_year_transient      = config['start_year_transient']
end_year_transient        = config['end_year_transient']

In [ ]:
outputs={}

## read raw data

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import h5py as h5

import scipy.signal
from datetime import datetime, timedelta
import pandas as pd

In [ ]:
#watershed_name = 'OakCreek'
#site_name = 'NF01'

data_raw_dir = './OakCreek_from_sundar'

fname_headbc_raw = os.path.join(data_raw_dir, 'bc_startend_raw.h5')

In [ ]:
# Read datasets from HDF5 file
with h5.File(fname_headbc_raw, "r") as hdf:
    time = hdf["Time"][:]  # Time array
    startpt_head = hdf["startpt_head"][:]  # Start point head data
    endpt_head = hdf["endpt_head"][:]  # End point head data

# Print shapes to confirm
print("Time shape:", time.shape)
print("Start point head shape:", startpt_head.shape)
print("End point head shape:", endpt_head.shape)

In [ ]:
# Create a plot with two lines
plt.figure(figsize=(10, 5))

plt.plot(time, startpt_head, label="Start Point Head", linestyle="-", color="blue")
plt.plot(time, endpt_head, label="End Point Head", linestyle="--", color="red")

# Formatting the plot
plt.xlabel("Time [days]")
plt.ylabel("Head Value")
plt.title("Start and End Point Head over Time")
plt.legend()
plt.grid(True)

# Show the plot
plt.show()

# Extract data
- for spinup and transient simulations

In [ ]:
# 11224 = 274 + 365*30 -> 2010/10/1
# 16059 = 364 + 365*43 -> 2023/12/31

# mask raw data for 1) steady-state spinup and cyclic spinup, 2) transient
## spinup
startdate_spinup_value = 11224 - 274 + 365 * (start_year_spinup - 2010)
enddate_spinup_value   = 11224 - 274 + 365 * (end_year_spinup - 2010) + 364
startdate_spinup       = f"{start_year_spinup}-01-01"
enddate_spinup         = f"{end_year_spinup}-12-31"
mask_spinup            = (time >= startdate_spinup_value) & (time <= enddate_spinup_value) # Create mask

time_spinup         = time[mask_spinup]
startpt_head_spinup = startpt_head[mask_spinup]
endpt_head_spinup   = endpt_head[mask_spinup]

fulldate_spinup = pd.date_range(start=startdate_spinup, end=enddate_spinup, freq='D')
datetime_spinup = fulldate_spinup[~((fulldate_spinup.month == 2) & (fulldate_spinup.day == 29))]
print(datetime_spinup)

## transient
startdate_transient_value = 11224 - 274 + 365 * (start_year_transient - 2010)
enddate_transient_value   = 11224 - 274 + 365 * (end_year_transient - 2010) + 364
startdate_transient       = f"{start_year_transient}-01-01"
enddate_transient         = f"{end_year_transient}-12-31"
mask_transient            = (time >= startdate_transient_value) & (time <= enddate_transient_value) # Create mask

time_transient         = time[mask_transient]
startpt_head_transient = startpt_head[mask_transient]
endpt_head_transient   = endpt_head[mask_transient]

fulldate_transient = pd.date_range(start=startdate_transient, end=enddate_transient, freq='D')
datetime_transient = fulldate_transient[~((fulldate_transient.month == 2) & (fulldate_transient.day == 29))]
print(datetime_transient)

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(12,3))
ax = axes.flatten()

ax[0].plot(datetime_spinup, startpt_head_spinup, label="Left Boundary Head", linestyle="-", color="blue")
ax[0].plot(datetime_spinup, endpt_head_spinup, label="Right Boundary Head", linestyle="--", color="red")
ax[0].set_xlabel("Time")
ax[0].set_ylabel("Head Value")
ax[0].set_title("Flow BCs - spinup raw")

ax[1].plot(datetime_transient, startpt_head_transient, label="Left Boundary Head", linestyle="-", color="blue")
ax[1].plot(datetime_transient, endpt_head_transient, label="Right Boundary Head", linestyle="--", color="red")
ax[1].set_xlabel("Time")
ax[1].set_ylabel("Head Value")
ax[1].set_title("Flow BCs - transient raw")

plt.show()

## calculate typical year head BC for spinup

In [ ]:
# Define number of days in a typical year
days_in_year = 365

# Compute number of full years available
num_years = end_year_spinup - start_year_spinup + 1

# Reshape data to group values by year (only full years)
startpt_head_reshaped = startpt_head_spinup[:num_years * days_in_year].reshape(num_years, days_in_year)
endpt_head_reshaped = endpt_head_spinup[:num_years * days_in_year].reshape(num_years, days_in_year)

# Compute the mean across years for each day of a typical year
startpt_head_typical = np.mean(startpt_head_reshaped, axis=0)
endpt_head_typical = np.mean(endpt_head_reshaped, axis=0)

In [ ]:
# Create an array for days in a typical year
typical_days = np.arange(365)

# Plot results
plt.figure(figsize=(6, 3))
plt.plot(typical_days, startpt_head_typical, label="Start Point Head (Typical Year)", color="blue")
plt.plot(typical_days, endpt_head_typical, label="End Point Head (Typical Year)", color="red", linestyle="--")

# Formatting
plt.xlabel("Day of the Year")
plt.ylabel("Head Value")
plt.title("Average Start and End Point Head for a Typical Year")
plt.legend()
#plt.grid(True)

# Show plot
plt.show()

In [ ]:
# Smoothing function from watershed-workflow [to-do, direct import from watershed-workflow?]
def smooth_array(data, method, axis=0, **kwargs):
    """Smooths fixed-interval time-series data using a Sav-Gol filter from scipy."""
    if method is True:
        method = 'savgol_filter'

    if method == 'savgol_filter':
        if 'window_length' not in kwargs:
            kwargs['window_length'] = 61  # Default window length for Savitzky-Golay filter
        if 'polyorder' not in kwargs:
            kwargs['polyorder'] = 2  # Default polynomial order for Savitzky-Golay filter
        if 'mode' not in kwargs:
            kwargs['mode'] = 'wrap'  # Wrap around for cyclic behavior
        return scipy.signal.savgol_filter(data, axis=axis, **kwargs)
    elif method == 'convolve':
        if 'window' not in kwargs:
            kwargs['window'] = 'hann'
        if 'Nx' not in kwargs:
            kwargs['Nx'] = 50
        win = scipy.signal.windows.get_window(**kwargs)
        win = win / win.sum()
        assert (len(data.shape) == 3 and axis == 0)
        data_new = np.empty_like(data)
        for i in range(data.shape[1]):
            for j in range(data.shape[2]):
                data_new[:, i, j] = scipy.signal.convolve(data[:, i, j],
                                                          win)[len(win) // 2:-len(win) // 2 + 1]
        return data_new
    else:
        raise ValueError(f'Invalid smooth method {method}')

In [ ]:
# Define smoothing parameters (from watershed-workflow)
smooth_kwargs = dict(window_length=181, polyorder=2)
#nyears_cyclic_spinup = 10

# Repeat for nyears_cyclic_spinup
startpt_head_repeated = np.tile(startpt_head_typical, nyears_cyclic_spinup)
endpt_head_repeated = np.tile(endpt_head_typical, nyears_cyclic_spinup)
typical_days_repeated = np.arange(365 * nyears_cyclic_spinup)

# Apply the smoothing
startpt_head_smooth_repeated = smooth_array(startpt_head_repeated, method='savgol_filter', window_length=181, polyorder=2, mode='wrap')
endpt_head_smooth_repeated = smooth_array(endpt_head_repeated, method='savgol_filter', window_length=181, polyorder=2, mode='wrap')

# Plot results
plt.figure(figsize=(10, 5))
plt.plot(typical_days_repeated, startpt_head_smooth_repeated, label="Start Point Head (Typical Year, Smoothed)", color="blue")
plt.plot(typical_days_repeated, endpt_head_smooth_repeated, label="End Point Head (Typical Year, Smoothed)", color="red", linestyle="--")

# Formatting
plt.xlabel("Day of the Year")
plt.ylabel("Head Value")
plt.title("Average Start and End Point Head for a Typical Year (Smoothed)")
plt.legend()
plt.grid(True)

# Show plot
plt.show()

## generate head BC for run0, run1, and run2

In [ ]:
# for run0, use mean head to drive steady-state spinup
startpt_head_smooth_repeated_mean = np.mean(startpt_head_smooth_repeated)
endpt_head_smooth_repeated_mean = np.mean(endpt_head_smooth_repeated)

print(startpt_head_smooth_repeated_mean)
print(endpt_head_smooth_repeated_mean)

In [ ]:
# for run1, use head typical year to drive cyclic spinup
## adjust the start_t to align with [cycle driver][start time]
## adjust the unit of time from day -> second
start_time_run1 = 0 # unit day, initially used to adjust the real start time
time_run1 = (start_time_run1 + np.arange(365 * nyears_cyclic_spinup))*24*3600 # unit second

outputs['BChead_start_spinup_filename_site'] = f'../data-processed/{site_name}/startpt_head_{nyears_cyclic_spinup}y_typical.h5'
with h5.File(outputs['BChead_start_spinup_filename_site'], "w") as hdf:
    hdf.create_dataset("Time", data=time_run1)
    hdf.create_dataset("startpt_head", data=startpt_head_smooth_repeated)
print(f"Data successfully written to {outputs['BChead_start_spinup_filename_site']}")

outputs['BChead_end_spinup_filename_site'] = f'../data-processed/{site_name}/endpt_head_{nyears_cyclic_spinup}y_typical.h5'
with h5.File(outputs['BChead_end_spinup_filename_site'], "w") as hdf:
    hdf.create_dataset("Time", data=time_run1)
    hdf.create_dataset("endpt_head", data=endpt_head_smooth_repeated)
print(f"Data successfully written to {outputs['BChead_end_spinup_filename_site']}")

In [ ]:
# for run2
## adjust the start_t to align with [cycle driver][start time]
## adjust the unit of time from day -> second
start_time_run2 = 0 # unit day, initially used to adjust the real start time
end2start_days_run2 = np.size(startpt_head_transient)
time_run2 = (start_time_run2 + np.arange(end2start_days_run2))*24*3600 # unit second

outputs['BChead_start_transient_filename_site'] = f'../data-processed/{site_name}/startpt_head_{startdate_transient}_{enddate_transient}.h5'
with h5.File(outputs['BChead_start_transient_filename_site'], "w") as hdf:
    hdf.create_dataset("Time", data=time_run2)
    hdf.create_dataset("startpt_head", data=startpt_head_transient)
print(f"Data successfully written to {outputs['BChead_start_transient_filename_site']}")

outputs['BChead_end_transient_filename_site'] = f'../data-processed/{site_name}/endpt_head_{startdate_transient}_{enddate_transient}.h5'
with h5.File(outputs['BChead_end_transient_filename_site'], "w") as hdf:
    hdf.create_dataset("Time", data=time_run2)
    hdf.create_dataset("endpt_head", data=endpt_head_transient)
print(f"Data successfully written to {outputs['BChead_end_transient_filename_site']}")

In [ ]:
# merged data for ats-pflotran transient, which restarted from ats-pflotran spinup run

In [ ]:
# Merge time
time_step = time_run1[1] - time_run1[0]  # Should be 86400 seconds
time_offset = time_run1[-1] + time_step
merged_time = np.concatenate((time_run1, time_run2 + time_offset))

# startpt_head_merged_spinup10yr_transient.h5
outputs['BChead_start_merge_spinup_transient_filename_site'] = f'../data-processed/{site_name}/startpt_head_merged_spinup{nyears_cyclic_spinup}yr_transient.h5'
outputs['BChead_end_merge_spinup_transient_filename_site'] = f'../data-processed/{site_name}/endpt_head_merged_spinup{nyears_cyclic_spinup}yr_transient.h5'

# 3. Merge and save the 'startpt_head' data
print(f"Merging 'startpt_head' data...")
merged_start_data = np.concatenate((startpt_head_smooth_repeated, startpt_head_transient))

with h5.File(outputs['BChead_start_merge_spinup_transient_filename_site'], "w") as hdf:
    hdf.create_dataset("Time", data=merged_time)
    hdf.create_dataset("startpt_head", data=merged_start_data)

print(f"Data successfully written to {outputs['BChead_start_merge_spinup_transient_filename_site']}")

# 4. Merge and save the 'endpt_head' data
print(f"Merging 'endpt_head' data...")
merged_end_data = np.concatenate((endpt_head_smooth_repeated, endpt_head_transient))

with h5.File(outputs['BChead_end_merge_spinup_transient_filename_site'], "w") as hdf:
    hdf.create_dataset("Time", data=merged_time)
    hdf.create_dataset("endpt_head", data=merged_end_data)

print(f"Data successfully written to {outputs['BChead_end_merge_spinup_transient_filename_site']}")

In [ ]:
outputs